In [1]:
import os
import tempfile
import logging
from urllib.parse import unquote
from pathlib import Path
from dotenv import load_dotenv

# Docling imports
from docling.document_converter import DocumentConverter

# LangChain imports
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_openai import AzureOpenAIEmbeddings
from langchain_community.vectorstores import AzureSearch

# Azure Storage imports
from azure.storage.blob import BlobServiceClient
from azure.identity import DefaultAzureCredential

/home/matrix4284/miniconda3/envs/azure_rag_app/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
with open("LC-GAN.pdf", "rb") as f:
    header = f.read(4)
    print(f"File header bytes: {header}")
    if header != b"%PDF":
        print("WARNING: File does not start with %PDF. It might not be a valid PDF.")

# 2. Parse with Docling
print("Parsing PDF with Docling (converting to Markdown)...")
print("PDF_PATH")
print(str("LC-GAN.pdf"))
converter = DocumentConverter()
result = converter.convert(str("LC-GAN.pdf"))
markdown_content = result.document.export_to_markdown()

# 3. Chunk with LangChain (Markdown Header Splitting)
print("Chunking document with MarkdownHeaderTextSplitter...")
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
chunks = markdown_splitter.split_text(markdown_content)
            
# Add metadata (source URL) to each chunk
for chunk in chunks:
    chunk.metadata["source"] = "this is test"
    print(chunk.metadata)

            
            
print(f"Created {len(chunks)} chunks.")

[INFO] 2026-03-09 22:43:33,667 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-09 22:43:33,672 [RapidOCR] download_file.py:60: File exists and is valid: /home/matrix4284/miniconda3/envs/azure_rag_app/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-09 22:43:33,672 [RapidOCR] main.py:53: Using /home/matrix4284/miniconda3/envs/azure_rag_app/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-09 22:43:33,763 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-09 22:43:33,765 [RapidOCR] download_file.py:60: File exists and is valid: /home/matrix4284/miniconda3/envs/azure_rag_app/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-09 22:43:33,765 [RapidOCR] main.py:53: Using /home/matrix4284/miniconda3/envs/azure_rag_app/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-09 22:43:33,809 [RapidO

File header bytes: b'%PDF'
Parsing PDF with Docling (converting to Markdown)...
PDF_PATH
LC-GAN.pdf


[INFO] 2026-03-09 22:43:33,820 [RapidOCR] download_file.py:60: File exists and is valid: /home/matrix4284/miniconda3/envs/azure_rag_app/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_rec_infer.onnx
[INFO] 2026-03-09 22:43:33,821 [RapidOCR] main.py:53: Using /home/matrix4284/miniconda3/envs/azure_rag_app/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_rec_infer.onnx
[WARNING] 2026-03-09 22:43:39,470 [RapidOCR] main.py:125: The text detection result is empty
RapidOCR returned empty result!
[WARNING] 2026-03-09 22:43:45,313 [RapidOCR] main.py:125: The text detection result is empty
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!


Chunking document with MarkdownHeaderTextSplitter...
{'source': 'this is test'}
{'Header 2': 'I. INTRODUCTION', 'source': 'this is test'}
{'Header 2': 'B. Applications in Industry:', 'source': 'this is test'}
{'Header 2': '2) CartoonGan:', 'source': 'this is test'}
{'Header 2': 'A. Lean Comic Gan (LC-GAN):', 'source': 'this is test'}
{'Header 2': '4) Vanilla Batch Normalization Process:', 'source': 'this is test'}
{'Header 2': '5) PitFalls of Batch Normalization:', 'source': 'this is test'}
{'Header 2': '7) Use of parameteric ReLU as learned rectifier:', 'source': 'this is test'}
{'Header 2': 'D. LC-GAN Architecture and Training Strategy:', 'source': 'this is test'}
{'Header 2': '1) Teacher Model Architecture:', 'source': 'this is test'}
{'Header 2': '2) Student Model Architecture:', 'source': 'this is test'}
{'Header 2': '3) Adversarial Model Architecture:', 'source': 'this is test'}
{'Header 2': 'E. Loss Function:', 'source': 'this is test'}
{'Header 2': '(a) Model without Distilled 

In [11]:
for chunk in chunks:
    print(chunk)

page_content='Lean Comic Gan (LC-GAN): A Light-Weight GAN-Based Architecture Leveraging Factorized Convolution by Depthwise Separable, Pointwise CNN and Teacher Model Forced Distilled Fetch Forw...  
<!-- image -->  
Lean Comic Gan (LC-GAN): A Light-Weight GAN-Based Architecture Leveraging Factorized Convolution by Depthwise Separable, Pointwise CNN and Teacher Model Forced Distilled Fetch Forward Perceptual Style Loss Aimed to Capture Two Dimensional Animated Style Photos using Smartphone Camera and Edge Devices  
Kaustav Mukherjee  
Abstract -In this paper, we propose a solution to design a lightweight GAN-based filter for smartphone cameras, Raspberry PI 4 equipped with PiCam and similar edge devices. This GAN-based camera filter will carry out neural style transfer to input images and impart a 2D animated comic movie-style look and texture to it. This will help the 2D animation artist to design or create new characters from real life person's images or even create scenes from real 

In [17]:
from processor import download_blob_to_bytes,pdf_markdown_parser
pdf_bytes=download_blob_to_bytes("https://kaustavst.blob.core.windows.net/uploads-2/LC-GAN.pdf")

Starting download for: https://kaustavst.blob.core.windows.net/uploads-2/LC-GAN.pdf
Downloaded data size: 1876932 bytes
File header bytes: b'%PDF'


In [19]:
markdown_text=pdf_markdown_parser(pdf_bytes)

Error parsing PDF: BaseModel.__init__() takes 1 positional argument but 2 were given
